# Car Dataset Demo with Ollama

This notebook runs `aryaman/dataset_generator/generate_car_dataset.py` while swapping in `aryaman/ollama_client.py` at import time, so the existing generator logic can run unchanged.

Before running it, make sure:

- your Ollama server is running
- the model in `OLLAMA_MODEL` is available locally
- the Python environment has the packages from `chunkingEval/dataset_generator/requirements.txt` installed


In [ ]:
from pathlib import Path
import os
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "aryaman" / "dataset_generator" / "generate_car_dataset.py").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find the repo root containing aryaman/dataset_generator/generate_car_dataset.py"
    )


repo_root = find_repo_root(Path.cwd().resolve())
chunking_root = repo_root / "aryaman"
generator_dir = chunking_root / "dataset_generator"
demo_root = generator_dir / "demo"

print(f"repo_root={repo_root}")
print(f"chunking_root={chunking_root}")
print(f"generator_dir={generator_dir}")
print(f"demo_root={demo_root}")


In [ ]:
# Adjust these defaults if your Ollama server or model uses different values.
os.environ.setdefault("OLLAMA_BASE_URL", "http://127.0.0.1:11434")
os.environ.setdefault("OLLAMA_MODEL", "llama3:8b")
os.environ.setdefault("OLLAMA_USER", "notebook-demo")

for key in ("OLLAMA_BASE_URL", "OLLAMA_MODEL", "OLLAMA_USER"):
    print(f"{key}={os.environ[key]}")


In [ ]:
import importlib.util


def load_module(module_name: str, path: Path):
    spec = importlib.util.spec_from_file_location(module_name, path)
    if spec is None or spec.loader is None:
        raise ImportError(f"Could not load module {module_name} from {path}")
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module


if str(chunking_root) not in sys.path:
    sys.path.insert(0, str(chunking_root))
if str(generator_dir) not in sys.path:
    sys.path.insert(0, str(generator_dir))

# Alias the Ollama shim as `client` before importing the generator script.
ollama_module = load_module("client", chunking_root / "dataset_generator" /"ollama_client.py")
sys.modules.pop("generate_car_dataset", None)
generator = load_module("generate_car_dataset", generator_dir / "generate_car_dataset.py")

client, model_name, request_user = ollama_module.load_client()
print(f"Loaded Ollama client for model '{model_name}' as user '{request_user}'.")
print("Generator module imported successfully.")


In [ ]:
# Keep the demo small at first, then increase `count` once everything is working.
count = 10
workers = 2
overwrite = False
max_retries = 3
delay_seconds = 2.0

selected_cars = generator.CAR_MODELS[:count]
expected_brand_count = len({brand for brand, _ in selected_cars})
car_output_dir = demo_root / "generated_cars"
brand_output_dir = demo_root / "generated_brands"

print(f"requested car count={count}")
print(f"expected brand file count={expected_brand_count}")
print(f"workers={workers}")
print(f"car_output_dir={car_output_dir}")
print(f"brand_output_dir={brand_output_dir}")
print("Selected cars:", selected_cars)


In [ ]:
car_output_dir.mkdir(parents=True, exist_ok=True)
brand_output_dir.mkdir(parents=True, exist_ok=True)

generator.generate_dataset(
    output_dir=car_output_dir,
    brand_output_dir=brand_output_dir,
    overwrite=overwrite,
    max_retries=max_retries,
    delay_seconds=delay_seconds,
    count=count,
    workers=workers,
)

print(f"Done. Car files written to: {car_output_dir}")
print(f"Done. Brand files written to: {brand_output_dir}")


In [ ]:
car_files = sorted(car_output_dir.glob("*.txt"))
brand_files = sorted(brand_output_dir.glob("*.txt"))

print(f"Requested {count} car files.")
print(f"Generated {len(car_files)} car files and {len(brand_files)} brand files.")
print("Car files:", [path.name for path in car_files[:5]])
print("Brand files:", [path.name for path in brand_files[:5]])

if len(car_files) != count:
    print(f"Warning: expected {count} car files but found {len(car_files)}.")
if len(brand_files) != expected_brand_count:
    print(f"Warning: expected {expected_brand_count} brand files but found {len(brand_files)}.")

if car_files:
    preview_lines = car_files[0].read_text(encoding="utf-8").splitlines()[:20]
    print(f"\nPreview from {car_files[0].name}:\n")
    print("\n".join(preview_lines))
else:
    print("No car files were generated yet.")


In [ ]:
jsonl_builder = load_module("build_jsonl_dataset", generator_dir / "build_jsonl_dataset.py")
qa_output_path = demo_root / "car_qa_dataset.jsonl"

car_models = jsonl_builder.load_car_models(jsonl_builder.GENERATOR_PATH)
brand_slug_to_name, car_slug_to_names = jsonl_builder.build_brand_lookup(car_models)
heading_lookup = jsonl_builder.build_alias_lookup()

car_paths = sorted(car_output_dir.glob("*.txt"))
brand_paths = sorted(brand_output_dir.glob("*.txt"))
all_paths = [(path, "car") for path in car_paths] + [(path, "brand") for path in brand_paths]

document_records = []
for path, doc_type in all_paths:
    title, sections, raw_text = jsonl_builder.parse_sections(path, doc_type, heading_lookup)
    document_records.append(
        jsonl_builder.build_document_record(
            path=path,
            doc_type=doc_type,
            title=title,
            sections=sections,
            raw_text=raw_text,
            brand_slug_to_name=brand_slug_to_name,
            car_slug_to_names=car_slug_to_names,
        )
    )

qa_records = jsonl_builder.build_qa_records(document_records)
jsonl_builder.validate_document_records(document_records, expected_count=len(all_paths))
jsonl_builder.validate_qa_records(qa_records)
jsonl_builder.write_qa_jsonl(qa_output_path, qa_records)
jsonl_builder.validate_jsonl_file(qa_output_path)

print(f"Wrote {len(qa_records)} QA rows to {qa_output_path}")
preview_lines = qa_output_path.read_text(encoding="utf-8").splitlines()[:3]
print("\nPreview lines:\n")
print("\n".join(preview_lines))